<a href="https://colab.research.google.com/github/hamed7pix/CSV2SQLWizard/blob/main/Serialize_Recreate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**To recreate a tree exactly as it was, you need a way to tell where the branches end. A standard pre-order traversal (e.g., 4, 2, 1...) isn't enough on its own because it doesn't capture the structural shape—you wouldn't know if 1 is a left child, a right child, or a leaf.**

**The solution is to record the null (empty) children using a special marker, like # or None.**

**If we serialize a tree using pre-order (Node -> Left -> Right) and include markers for empty children, the first item in our list is always the root. We can recursively rebuild the tree by popping elements off the list.**

In [4]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def serialize(root):
    result = []
    def preorder(node):
        if node is None:
            result.append("#")
            return
        result.append(str(node.val))
        preorder(node.left)
        preorder(node.right)

    preorder(root)
    return ",".join(result)

def deserialize(data):
    values = data.split(",")
    iterator = iter(values)

    def build_tree():
        try:
            val = next(iterator)
        except StopIteration:
            return None

        if val == "#":
            return None

        node = TreeNode(int(val))
        node.left = build_tree()
        node.right = build_tree()
        return node

    return build_tree()

In [5]:
# --- NEW 2D VISUAL HELPER FUNCTION ---
def print_tree(root):
    """Prints the tree visually in a 2D ASCII format"""
    def get_height(node):
        if not node: return 0
        return 1 + max(get_height(node.left), get_height(node.right))

    h = get_height(root)
    if h == 0:
        return

    # Determine the size of the 2D grid based on tree height
    cols = 2 ** (h + 1)
    rows = 2 * h - 1
    grid = [[" " for _ in range(cols)] for _ in range(rows)]

    def place(node, row, col, level):
        if not node:
            return

        # 1. Place the node's value in the grid
        val_str = str(node.val)
        for i, char in enumerate(val_str):
            if col + i < cols:
                grid[row][col + i] = char

        # 2. Place the branch slashes and child nodes
        if level < h - 1:
            # dx determines how wide the branches spread based on current level
            dx = 2 ** (h - level - 1)
            if node.left:
                grid[row + 1][col - dx // 2] = "/"
                place(node.left, row + 2, col - dx, level + 1)
            if node.right:
                grid[row + 1][col + dx // 2] = "\\"
                place(node.right, row + 2, col + dx, level + 1)

    # Place root node with dynamic center spacing based on height
    place(root, 0, 2 ** h - 1, 0)

    # Print the resulting grid
    print("\nVisual Tree Output:")
    for r in grid:
        line = "".join(r).rstrip()
        if line: # Only print lines that have content
            print(line)


# ==========================================
# TEST WITH THE INPUT TREE
# ==========================================
root = TreeNode(4)
root.left = TreeNode(2)
root.right = TreeNode(6)

root.left.left = TreeNode(1)
root.left.right = TreeNode(3)

root.right.left = TreeNode(5)
root.right.right = TreeNode(7)

serialized_string = serialize(root)
recreated_root = deserialize(serialized_string)

# Print the recreated tree using the new visual formatter
print_tree(recreated_root)


Visual Tree Output:
       4
     /   \
   2       6
  / \     / \
 1   3   5   7
